# Pipeline 6 — Resident Incident Risk (Next 30 Days)

**Created:** 2026-04-06T23:33:27.174933Z

This notebook follows **CRISP-DM** while also satisfying the IS455 rubric sections:
- Problem Framing
- Data Acquisition, Preparation & Exploration
- Modeling & Feature Selection
- Evaluation & Interpretation
- Causal and Relationship Analysis
- Deployment Notes


## CRISP-DM Overview

### 1) Business Understanding
Goal: flag residents at higher incident risk so staff can intervene earlier.

### 2) Data Understanding
Join resident master data with incident reports, home visits, process recordings, education and health records.

### 3) Data Preparation
Define cutoff date and compute recent-history features (30/90 days) with latest monthly records.

### 4) Modeling
Predictive: Gradient Boosting Classifier. Explanatory: Logistic Regression.

### 5) Evaluation
ROC-AUC/F1 and discuss false positives/negatives in a sensitive setting.

### 6) Deployment
Export resident risk scores and bands; import into `/api/ml/import` and view in `/app/ml`.


## 1) Problem Framing (Rubric)

State:
- the business question,
- who cares,
- why it matters,
- predictive vs explanatory goals.

We build **two models**:
- Predictive (optimize out-of-sample performance)
- Explanatory (interpretability / relationship analysis)


## 2) Data Acquisition, Preparation & Exploration (Rubric)

Rules to avoid leakage:
- Define an **as-of date** (cutoff).
- Build features using only data **on or before** the cutoff.
- Create labels using only data **after** the cutoff in a defined horizon.


In [ ]:
import json
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor


REPO_ROOT = Path("..").resolve()
RAW_DIR_A = (REPO_ROOT / "data" / "raw" / "lighthouse_csv_v7").resolve()
RAW_DIR_B = (REPO_ROOT / "data" / "raw").resolve()
DATA_DIR = RAW_DIR_A if RAW_DIR_A.exists() else RAW_DIR_B

OUT_DIR = (REPO_ROOT / "output" / "ml-predictions").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Out dir:", OUT_DIR)


def require_csv(stem: str) -> pd.DataFrame:
    path = DATA_DIR / f"{stem}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}.")
    return pd.read_csv(path, encoding="utf-8-sig")


# After POST /api/admin/lighthouse-import, train from Azure SQL by setting INTEX_ODBC to your ODBC connection string.
SQL_TABLE_BY_STEM = {
    "supporters": "Supporters",
    "donations": "Contributions",
    "social_media_posts": "SocialMediaPosts",
    "safehouse_monthly_metrics": "SafehouseMonthlyMetrics",
    "residents": "Residents",
    "incident_reports": "IncidentReports",
    "home_visitations": "HomeVisitations",
    "process_recordings": "ProcessRecordings",
    "education_records": "EducationRecords",
    "health_wellbeing_records": "HealthWellbeingRecords",
}


def load_df(stem: str) -> pd.DataFrame:
    odbc = os.environ.get("INTEX_ODBC")
    table = SQL_TABLE_BY_STEM.get(stem)
    if odbc and table:
        try:
            import pyodbc

            with pyodbc.connect(odbc, timeout=120) as cnx:
                df = pd.read_sql(f"SELECT * FROM [{table}]", cnx)
            print(f"DB [{table}] rows:", len(df))
            if len(df) > 0:

                def pascal_to_snake(name: str) -> str:
                    return re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()

                df = df.rename(columns={c: pascal_to_snake(str(c)) for c in df.columns})
                if stem == "donations":
                    df = df.rename(columns={"contribution_id": "donation_id"})
                if stem == "process_recordings":
                    df = df.rename(columns={"process_recording_id": "recording_id"})
                if stem == "home_visitations":
                    df = df.rename(columns={"home_visitation_id": "visitation_id"})
                return df
        except Exception as ex:
            print("INTEX_ODBC failed, using CSV:", ex)
    return require_csv(stem)


def to_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.date


def to_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", utc=True)


def eval_classification(y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}
    if y_proba is not None:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba))
        except Exception:
            pass
    return out


def eval_regression(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred, squared=False)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def export_predictions_json(prediction_type: str, entity_type: str, df_out: pd.DataFrame, id_col: str, score_col: str, label_col: str | None = None):
    out_path = OUT_DIR / f"{prediction_type}.json"
    rows = []
    for _, r in df_out.iterrows():
        rows.append(
            {
                "predictionType": prediction_type,
                "entityType": entity_type,
                "entityId": int(r[id_col]),
                "score": float(r[score_col]),
                "label": None if label_col is None else (None if pd.isna(r[label_col]) else str(r[label_col])),
                "payloadJson": json.dumps({k: v for k, v in r.items() if k not in {id_col, score_col, label_col}}, default=str),
            }
        )
    out_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print("Wrote:", out_path, "rows=", len(rows))


In [ ]:
residents = load_df("residents")
        incidents = load_df("incident_reports")
        visits = load_df("home_visitations")
        recordings = load_df("process_recordings")
        edu = load_df("education_records")
        health = load_df("health_wellbeing_records")

incidents["incident_date"] = pd.to_datetime(incidents["incident_date"], errors="coerce")
visits["visit_date"] = pd.to_datetime(visits["visit_date"], errors="coerce")
recordings["session_date"] = pd.to_datetime(recordings["session_date"], errors="coerce")
edu["record_date"] = pd.to_datetime(edu["record_date"], errors="coerce")
health["record_date"] = pd.to_datetime(health["record_date"], errors="coerce")

max_incident = incidents["incident_date"].max()
cutoff = max_incident - pd.Timedelta(days=30)
label_end = cutoff + pd.Timedelta(days=30)
print("Cutoff:", cutoff.date(), "Label end:", label_end.date())

# Label: any incident in next 30 days
future_inc = incidents[(incidents["incident_date"] > cutoff) & (incidents["incident_date"] <= label_end)]
y = (future_inc.groupby("resident_id")["incident_id"].count() > 0).astype(int).rename("incident_next_30d").reset_index()

past_inc = incidents[incidents["incident_date"] <= cutoff].copy()
past_vis = visits[visits["visit_date"] <= cutoff].copy()
past_rec = recordings[recordings["session_date"] <= cutoff].copy()
past_edu = edu[edu["record_date"] <= cutoff].copy()
past_h = health[health["record_date"] <= cutoff].copy()

base = residents[["resident_id","safehouse_id","case_status","case_category","is_pwd","family_is_4ps","family_solo_parent","family_indigenous","reintegration_status"]].copy()
for b in ["is_pwd","family_is_4ps","family_solo_parent","family_indigenous"]:
    base[b] = base[b].astype(int)

# Incident history features
inc_90 = past_inc[past_inc["incident_date"] >= (cutoff - pd.Timedelta(days=90))]
inc_feat = inc_90.groupby("resident_id").agg(
    incidents_90d=("incident_id","count"),
    high_sev_90d=("severity", lambda s: int((s == "High").sum()))
).reset_index()

# Home visit follow-up features
vis_90 = past_vis[past_vis["visit_date"] >= (cutoff - pd.Timedelta(days=90))]
vis_feat = vis_90.groupby("resident_id").agg(
    visits_90d=("visitation_id","count"),
    followups_90d=("follow_up_needed", lambda s: int((s == True).sum()))
).reset_index()

# Process recording volume features
rec_30 = past_rec[past_rec["session_date"] >= (cutoff - pd.Timedelta(days=30))]
rec_feat = rec_30.groupby("resident_id").agg(
    recordings_30d=("recording_id","count"),
    concerns_flagged_30d=("concerns_flagged", lambda s: int((s == True).sum()))
).reset_index()

def last_by_res(df, date_col, value_cols):
    df2 = df.dropna(subset=[date_col]).sort_values(["resident_id", date_col]).copy()
    last = df2.groupby("resident_id").tail(1)
    return last[["resident_id"] + value_cols]

edu_last = last_by_res(past_edu, "record_date", ["attendance_rate","progress_percent","completion_status","enrollment_status"])
h_last = last_by_res(past_h, "record_date", ["general_health_score","nutrition_score","sleep_quality_score","energy_level_score"])

df = base.merge(inc_feat, on="resident_id", how="left") \\\n+        .merge(vis_feat, on="resident_id", how="left") \\\n+        .merge(rec_feat, on="resident_id", how="left") \\\n+        .merge(edu_last, on="resident_id", how="left") \\\n+        .merge(h_last, on="resident_id", how="left") \\\n+        .merge(y, on="resident_id", how="left")

df["incident_next_30d"] = df["incident_next_30d"].fillna(0).astype(int)
for c in ["incidents_90d","high_sev_90d","visits_90d","followups_90d","recordings_30d","concerns_flagged_30d"]:
    df[c] = df[c].fillna(0)
for c in ["attendance_rate","progress_percent","general_health_score","nutrition_score","sleep_quality_score","energy_level_score"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(df[c].median() if df[c].notna().any() else 0)

print("Rows:", len(df), "Pos rate:", df["incident_next_30d"].mean())
df.head()


In [ ]:
target = "incident_next_30d"
features = [
    "safehouse_id","case_status","case_category",
    "is_pwd","family_is_4ps","family_solo_parent","family_indigenous",
    "incidents_90d","high_sev_90d","visits_90d","followups_90d","recordings_30d","concerns_flagged_30d",
    "attendance_rate","progress_percent","completion_status","enrollment_status",
    "general_health_score","nutrition_score","sleep_quality_score","energy_level_score"
]

X = df[features].copy()
y = df[target].copy()

cat_cols = ["case_status","case_category","completion_status","enrollment_status"]
num_cols = [c for c in features if c not in cat_cols and c != "safehouse_id"] + ["safehouse_id"]

pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


In [ ]:
gb = Pipeline(steps=[("pre", pre), ("model", GradientBoostingClassifier(random_state=42))])
gb.fit(X_train, y_train)
proba = gb.predict_proba(X_test)[:,1]
pred = (proba >= 0.5).astype(int)
print("Predictive (GB):", eval_classification(y_test, pred, proba))

lr = Pipeline(steps=[("pre", pre), ("model", LogisticRegression(max_iter=2500))])
lr.fit(X_train, y_train)
proba2 = lr.predict_proba(X_test)[:,1]
pred2 = (proba2 >= 0.5).astype(int)
print("Explanatory (LogReg):", eval_classification(y_test, pred2, proba2))


In [ ]:
# Score all residents
df_out = df[["resident_id"] + features].copy()
df_out["risk_score"] = gb.predict_proba(df_out[features])[:,1]
df_out["risk_band"] = pd.qcut(df_out["risk_score"], q=4, labels=["Low","Medium","High","Very High"])

export_predictions_json(
    prediction_type="resident_incident_30d",
    entity_type="Resident",
    df_out=df_out[["resident_id","risk_score","risk_band","safehouse_id","incidents_90d","followups_90d","recordings_30d","progress_percent","general_health_score"]].rename(columns={"risk_score":"score"}),
    id_col="resident_id",
    score_col="score",
    label_col="risk_band"
)

# Optional: readiness analysis (not exported; still useful for write-up)
df["ready"] = (df["reintegration_status"] == "Completed").astype(int)
print("Ready rate:", df["ready"].mean())


## 3) Modeling & Feature Selection (Rubric)

- Predictive model: tree/ensemble
- Explanatory model: linear/logistic regression


## 4) Evaluation & Interpretation (Rubric)

Interpret in business terms, and discuss real-world costs of errors.

## 5) Causal and Relationship Analysis (Rubric)

Discuss relationships, confounding risks, and where correlation ≠ causation.


## 6) Deployment Notes (Rubric)

Export predictions to JSON and import into the deployed app:
- `POST /api/ml/import?replace=true` (admin-only)
- View in `/app/ml` (Staff Portal → ML Insights)
